# Week 13: the corpus-overfit fix (more genuine speakers)

The deployed S2 adapts on only **21 genuine speakers**. This run trains the same recipe
on **248** (the 227 extra adaptation speakers, 3,405 genuine clips, plus their 1,759
CM08 XTTS clones; `src/data/extended_training.py`) and compares it with the deployed
recipe **seed for seed**. Nothing else changes: same dev set, Stage-1 start, learning
rate, epochs. The comparison is fixed in advance; no test set is used to choose
anything.

| Runs | Time on a T4 |
|---|---|
| extended recipe, seeds 1234, 2025, 7, 42, 99 | ~30 min each |
| deployed recipe, the same five seeds | ~10 min each |

Every model is scored over the phone line on: the original test pool, CM04 (Tortoise),
the RVC test half, the **60 new speakers** (CM05 XTTS and CM06 Tortoise), **CM07
IndicF5**, and the false alarms on the **language-control** genuine set.

**How to run.** Inputs (all four): `saikrishnareddy9/codemix-bundle-normalised`,
`saikrishnareddy9/codemix-extension-bundle` (the genuine clips of the new speakers),
`saikrishnareddy9/codemix-week13-clones` (the CM05-CM08 clones) and
`saikrishnareddy9/codemix-s3-native-results` (the Stage-1 checkpoint).
Settings: **GPU T4 x2** (one is used), Internet **On**. Then **Save Version -> Save & Run
All (Commit) -> Save**: it runs in the background, so the tab can be closed and the
laptop shut down.

**Getting the results.** `/kaggle/working` keeps only `overfit_fix_results.zip` and the
two seed-1234 checkpoints. When the version finishes: the notebook's page -> **Output**
-> `overfit_fix_results.zip` -> Download. The zip is rewritten after every run, and the
run stops itself at 10.5 hours, so a result is always saved.

## 1. Settings

In [ ]:
SEEDS = [1234, 2025, 7, 42, 99]
SYSTEMS = {  # name -> config; runs alternate so every finished pair is comparable
    "ext": "configs/train_lora_ext_rvc_channel.yaml",
    "deployed": "configs/train_lora_norm_rvc_channel.yaml",
}
EVALS = {  # EER sets, all over the phone line
    "eval_pool": "data/manifests/codemix_eval_channel20.csv",
    "cm04": "data/manifests/score_cm04_norm_channel20.csv",
    "rvc_test": "data/manifests/rvc_holdout_test_channel20.csv",
    "new_speakers": "data/manifests/score_ext_channel20.csv",
    "cm07": "data/manifests/score_cm07_channel20.csv",
}
CONTROL = "data/manifests/mucs_language_control_channel20.csv"  # genuine only: false alarms
ZIP_NAME = "overfit_fix_results.zip"
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH = "week13-mounika-overfit-fix"
SCORE_ARGS = "--device cuda --batch-size 32 --num-workers 4 --max-seconds 4.0"

## 2. Setup: clone, Stage-1 checkpoint, one data root, checks

In [ ]:
import glob, hashlib, json, os, re, shutil, subprocess, sys, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")    # the saved output: the zip and two checkpoints only
TEMP = Path("/kaggle/temp")
try:
    TEMP.mkdir(parents=True, exist_ok=True)
    (TEMP / ".writable").write_text("ok")
except OSError:
    TEMP = Path("/tmp/w13")
    TEMP.mkdir(parents=True, exist_ok=True)
REPO = TEMP / "codemix-deepfake-detection"
OUT = TEMP / "overfit_fix_results"
ROOT = TEMP / "root_channel"      # both datasets' phone-line clips, linked into one folder
DEADLINE = time.time() + 10.5 * 3600  # Kaggle stops a run at 12 h; leave time to save
FAILURES = []


def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True,
                       cwd=REPO if REPO.exists() else None)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode and r.stderr.strip():
        print(r.stderr[-2500:])
    if check and r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r


def run_logged(args):
    """Run a long command, streaming its output, and return all of it."""
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    proc = subprocess.Popen(args, text=True, errors="replace", cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    with open(OUT / "run_log.txt", "a", encoding="utf-8") as log:
        log.write(f"\n$ {' '.join(args)}\n")
        for line in proc.stdout:
            lines.append(line)
            log.write(line)
            if not any(n in line for n in ("Warning", "warn(", "Some weights", "You should")):
                print(line, end="", flush=True)
    proc.wait()
    return proc.returncode, "".join(lines)


def save_progress():
    """Zip the results so far (never checkpoints) into /kaggle/working."""
    import zipfile
    staging = WORK / (ZIP_NAME + ".tmp")
    with zipfile.ZipFile(staging, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(OUT.rglob("*")):
            if f.is_file() and f.suffix != ".pt":
                zf.write(f, f.relative_to(OUT.parent))
    staging.replace(WORK / ZIP_NAME)


def tail(text, n=30):
    return "\n".join([line for line in text.splitlines() if line.strip()][-n:])


def failed(where, detail):
    FAILURES.append({"where": where, "detail": detail})
    print(f"\n!!!!!! FAILED: {where}\n{detail}\n!!!!!! recorded; the run continues\n", flush=True)
    try:
        (OUT / "FAILURES.txt").write_text(
            "\n\n".join(f"== {f['where']}\n{f['detail']}" for f in FAILURES), encoding="utf-8")
        save_progress()
    except Exception as e:
        print("(could not write FAILURES.txt:", e, ")")


def _make_writable(func, path, _):
    os.chmod(path, 0o700)
    func(path)


os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
for attempt in range(1, 4):
    if REPO.exists():
        shutil.rmtree(REPO, onerror=_make_writable)
    if subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1",
                       f"https://{REPO_HOST}", str(REPO)]).returncode == 0:
        break
    print(f"clone attempt {attempt} failed; retrying in 20 s", flush=True)
    time.sleep(20)
else:
    raise SystemExit("Could not clone the repository after 3 attempts. Is Internet switched On?")
os.chdir(REPO)
sys.path.insert(0, str(REPO))
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, text=True,
                        capture_output=True).stdout.strip()

BASELINE = REPO / "checkpoints" / "baseline" / "best.pt"
pointer = subprocess.run(["git", "cat-file", "-p", "HEAD:checkpoints/baseline/best.pt"],
                         cwd=REPO, text=True, capture_output=True).stdout
want = re.search(r"oid sha256:([0-9a-f]{64})", pointer).group(1)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


source = None
for candidate in glob.glob("/kaggle/input/**/checkpoints/baseline/best.pt", recursive=True):
    if os.path.getsize(candidate) > 100_000_000 and sha256(candidate) == want:
        shutil.copy(candidate, BASELINE)
        source = candidate
        break
if source is None:
    print("Stage-1 checkpoint not in the inputs; fetching it with git lfs ...")
    sh("git lfs install --local && git lfs pull --include checkpoints/baseline/best.pt", check=False)
    source = "git lfs"
if not BASELINE.is_file() or BASELINE.stat().st_size < 100_000_000 or sha256(BASELINE) != want:
    raise SystemExit("The Stage-1 checkpoint could not be obtained. Attach "
                     "saikrishnareddy9/codemix-s3-native-results and run again.")
print(f"Stage-1 checkpoint OK (sha256 verified) from {source}")
sh(f"{sys.executable} -m pip install -q 'transformers>=4.57,<5' soundfile==0.12.1 librosa==0.11.0",
   check=False)

# One data root: every phone-line clip of the three datasets, linked (not copied) into ROOT/clips.
folders = [p for p in glob.glob("/kaggle/input/**/channel/clips", recursive=True) if os.listdir(p)]
# The clones dataset holds only phone-line clips, unpacked as clips/ at its top level.
folders += [p for p in glob.glob("/kaggle/input/**/clips", recursive=True)
            if "week13-clones" in p and p not in folders and os.listdir(p)]
print("phone-line clip folders:", folders)
if len(folders) < 3:
    raise SystemExit("Attach saikrishnareddy9/codemix-bundle-normalised, "
                     "saikrishnareddy9/codemix-extension-bundle and "
                     "saikrishnareddy9/codemix-week13-clones (Add Input).")
shutil.rmtree(ROOT, ignore_errors=True)
(ROOT / "clips").mkdir(parents=True)
linked = same_name = 0
for folder in folders:
    for name in os.listdir(folder):
        target = ROOT / "clips" / name
        if target.exists():
            same_name += 1
            continue
        os.symlink(os.path.join(folder, name), target)
        linked += 1
print(f"data root {ROOT}: {linked} clips linked ({same_name} names in both datasets, first kept)")

import pandas as pd
import yaml
from src.data.scoring_manifests import missing_files

manifests = set(EVALS.values()) | {CONTROL}
for config in SYSTEMS.values():
    cfg = yaml.safe_load((REPO / config).read_text())
    manifests |= {cfg["train_manifest"], cfg["dev_manifest"]}
for manifest in sorted(manifests):
    gone = missing_files(pd.read_csv(manifest), str(ROOT))
    print(f"  {manifest:58s} missing {len(gone)}")
    if gone:
        raise SystemExit(f"{manifest} has clips in no attached dataset, e.g. {gone[:3]} -- "
                         "are all three data inputs attached?")
print("every clip every run needs is present")
sh(f"{sys.executable} -m pytest tests/test_splits.py tests/test_config_guard.py "
   "tests/test_extended_training.py -q")
for config in SYSTEMS.values():
    sh(f"{sys.executable} -m src.training.config_guard {config}")

## 3. Train, score, summarise, and the download link

In [ ]:
import statistics as st
import zipfile

DEV_LINE = re.compile(r"best dev EER = ([0-9.]+)")
RUNS = []
DONE = False
FIELDS = ["eval_pool", "cm04", "rvc_test", "new_xtts", "new_tortoise", "cm07", "control_fa"]


def score(tag, ckpt, manifest):
    """EER JSON of one model on one set, or None (recorded as a failure)."""
    code_, text = run_logged([sys.executable, "-m", "src.training.evaluate", "--checkpoint", ckpt,
                              "--manifest", manifest, "--encoder", "wav2vec2-base",
                              "--data-root", ROOT, *SCORE_ARGS.split(),
                              "--scores-out", OUT / f"{tag}_scores.csv", "--out", OUT / f"{tag}.json"])
    if code_ == 0:
        try:
            return json.loads((OUT / f"{tag}.json").read_text())
        except Exception:
            pass
    failed(f"scoring {tag} (exit {code_})", tail(text))
    return None


def false_alarms(tag, ckpt):
    code_, text = run_logged([sys.executable, "-m", "src.training.false_alarm", "--checkpoint", ckpt,
                              "--manifest", CONTROL, "--encoder", "wav2vec2-base",
                              "--data-root", ROOT, "--device", "cuda", "--batch-size", "32",
                              "--num-workers", "4", "--scores-out", OUT / f"{tag}_scores.csv",
                              "--out", OUT / f"{tag}.json"])
    try:
        return json.loads((OUT / f"{tag}.json").read_text())["summary"]["false_alarm_rate"]
    except Exception:
        failed(f"false alarms {tag} (exit {code_})", tail(text))
        return None


def per_attack(result, attack):
    for row in (result or {}).get("per_attack", []):
        if row["attack"] == attack:
            return row["eer"]
    return None


def summary_text():
    lines = ["STATUS: " + ("FINISHED - every run ran." if DONE else
                           f"IN PROGRESS - {len(RUNS)} run(s) finished so far."),
             "RUN HEALTH: " + ("every step ran without error." if not FAILURES else
                               f"{len(FAILURES)} step(s) FAILED -- see FAILURES.txt"),
             f"commit: {COMMIT}", "",
             "EER (%) per run, phone line; control_fa = genuine clips called fake (%)",
             f"{'run':18s}{'status':>9s}{'dev':>7s}" + "".join(f"{f:>13s}" for f in FIELDS)]
    for r in RUNS:
        dev = "-" if r["dev_eer"] is None else f"{100 * r['dev_eer']:.2f}"
        cells = "".join(f"{'-' if r.get(f) is None else format(100 * r[f], '.2f'):>13s}" for f in FIELDS)
        lines.append(f"{r['name']:18s}{r['status']:>9s}{dev:>7s}{cells}")
    lines.append("\nMEAN +/- SD over seeds, and the paired difference ext - deployed (same seeds)")
    for f in FIELDS:
        parts = []
        for system in SYSTEMS:
            v = [100 * r[f] for r in RUNS if r["system"] == system and r.get(f) is not None]
            parts.append(f"{system} " + (f"{st.mean(v):6.2f} +/- {st.stdev(v):5.2f} (n={len(v)})"
                                         if len(v) >= 2 else ("-" if not v else f"{v[0]:6.2f} (n=1)")))
        by_seed = {}
        for r in RUNS:
            if r.get(f) is not None:
                by_seed.setdefault(r["seed"], {})[r["system"]] = 100 * r[f]
        diffs = [d["ext"] - d["deployed"] for d in by_seed.values() if len(d) == 2]
        paired = (f"diff {st.mean(diffs):+6.2f} (n={len(diffs)}, ext better in "
                  f"{sum(x < 0 for x in diffs)})" if diffs else "")
        lines.append(f"  {f:13s} " + "   ".join(parts) + "   " + paired)
    return "\n".join(lines)


def write_summary():
    try:
        text = summary_text()
    except Exception:
        text = "The summary could not be built:\n" + traceback.format_exc()
    try:
        (OUT / "summary.txt").write_text(text, encoding="utf-8")
        (OUT / "summary.json").write_text(json.dumps({
            "mode": "full" if DONE else "in progress", "commit": COMMIT, "branch": BRANCH,
            "seeds": SEEDS, "systems": SYSTEMS, "runs": RUNS, "failures": FAILURES},
            indent=2, default=str))
    except Exception as e:
        print("(could not write the summary:", e, ")", flush=True)
    return text


def show_download_link():
    from IPython.display import FileLink, display
    path = WORK / ZIP_NAME
    if not path.is_file():
        print("No results zip was written -- see the errors above.")
        return
    with zipfile.ZipFile(path) as zf:
        count, broken = len(zf.namelist()), zf.testzip()
    print(f"\n{ZIP_NAME}: {count} files, {path.stat().st_size / 1e6:.1f} MB,",
          "checked OK" if broken is None else f"DAMAGED at {broken}")
    os.chdir(WORK)
    print("\nCLICK THIS LINK TO DOWNLOAD THE RESULTS:")
    display(FileLink(ZIP_NAME))
    print(f"In a saved version: the notebook's page -> Output -> {ZIP_NAME} -> Download.")


def run_job(system, seed):
    """Train one system with one seed, score it everywhere, record and save the row."""
    name = f"{system}_s{seed}"
    t0 = time.time()
    row = {"name": name, "system": system, "seed": seed, "dev_eer": None, "status": "failed"}
    try:
        cfg = yaml.safe_load((REPO / SYSTEMS[system]).read_text())
        cfg.update({"seed": int(seed), "out_dir": f"checkpoints/w13/{name}", "wandb_mode": "disabled"})
        cfg_path = REPO / "configs" / f"w13_{name}.yaml"
        cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
        code_, text = run_logged([sys.executable, "-m", "src.training.train", "--config", cfg_path,
                                  "--data-root", ROOT, "--device", "cuda"])
        ckpt = REPO / cfg["out_dir"] / "best.pt"
        if code_ or not ckpt.is_file():
            row["status"] = "collapsed" if "model collapsed" in text else "failed"
            failed(f"training {name} (exit {code_})", tail(text))
            return row
        dev = DEV_LINE.findall(text)
        row["dev_eer"] = float(dev[-1]) if dev else None
        results = {ev: score(f"{name}__{ev}", ckpt, m) for ev, m in EVALS.items()}
        for ev in ("eval_pool", "cm04", "rvc_test", "cm07"):
            row[ev] = (results[ev] or {}).get("pooled", {}).get("eer")
        row["new_xtts"] = per_attack(results["new_speakers"], "xtts_v2")
        row["new_tortoise"] = per_attack(results["new_speakers"], "tortoise")
        row["control_fa"] = false_alarms(f"{name}__control", ckpt)
        row["status"] = "ok" if all(row.get(f) is not None for f in FIELDS) else "partial"
        if seed == SEEDS[0]:  # kept for later scoring on the laptop (HiACC, clean bundle)
            shutil.copy(ckpt, WORK / f"{name}_best.pt")
    except Exception:
        failed(f"{name}: unexpected error", traceback.format_exc())
    finally:
        shutil.rmtree(REPO / "checkpoints" / "w13" / name, ignore_errors=True)
        row["minutes"] = round((time.time() - t0) / 60, 1)
        RUNS.append(row)
        (OUT / "runs.json").write_text(json.dumps(RUNS, indent=2))
        write_summary()
        save_progress()
        shown = "  ".join(f"{f} {100 * row[f]:.2f}" for f in FIELDS if row.get(f) is not None)
        print(f"\n[{name}] {row['status']}  {shown}  ({row['minutes']} min)\n", flush=True)
    return row


try:
    jobs = [(system, seed) for seed in SEEDS for system in SYSTEMS]
    for system, seed in jobs:
        if time.time() > DEADLINE:
            failed("time limit", f"stopped before {system}_s{seed}; finished runs are kept")
            break
        run_job(system, seed)
    else:
        DONE = True
        print("\nALL RUNS FINISHED\n", flush=True)
finally:
    print(write_summary(), flush=True)
    save_progress()
    show_download_link()

## 4. Optional: the download link again

Section 3 already ends with the link. Run this only if you need it again.

In [ ]:
import os, zipfile
from pathlib import Path
from IPython.display import FileLink, display

os.chdir("/kaggle/working")
if Path(ZIP_NAME).is_file():
    with zipfile.ZipFile(ZIP_NAME) as zf:
        print(f"{ZIP_NAME}: {len(zf.namelist())} files, checked", "OK" if zf.testzip() is None else "DAMAGED")
    display(FileLink(ZIP_NAME))
else:
    print("No results zip yet -- section 3 writes it after its first finished run.")